# Flood Prediction Model
## File: models/train_flood.ipynb

**Run karo:** Kernel → Restart & Run All
**Output:** `flood_model.pkl` → Copy karo `backend/` folder mein

## Step 1 — Libraries

In [1]:
import numpy as np
import pandas as pd
import pickle
import warnings
warnings.filterwarnings('ignore')
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (classification_report, f1_score,
    roc_auc_score, confusion_matrix, roc_curve)
print("Ready!")

Ready!


## Step 2 — Dataset

In [2]:
def generate_flood_data(n=5000, seed=42):
    
    #Realistic flood dataset for Indian river basins.
    #Key factors: river level + heavy rain + poor drainage + low elevation
    
    #Real sources:
    #- CWC river levels: http://cwc.gov.in
    #- IMD rainfall: https://mausam.imd.gov.in
    
    np.random.seed(seed)
    
    rainfall_mm       = np.abs(np.random.normal(40, 35, n))
    river_level_m     = np.random.uniform(0.5, 12, n)
    drainage_capacity = np.random.uniform(0.1, 1.0, n)
    soil_saturation   = np.random.uniform(0.1, 1.0, n)
    upstream_flow     = np.random.uniform(10, 5000, n)
    elevation         = np.random.uniform(1, 500, n)
    slope             = np.random.uniform(0, 15, n)
    distance_river_km = np.random.exponential(2, n).clip(0.1, 20)
    past_rain_3day    = rainfall_mm * np.random.uniform(1.5, 5, n)
    past_rain_7day    = past_rain_3day * np.random.uniform(2, 6, n)
    humidity          = np.random.uniform(50, 100, n)
    month             = np.random.randint(1, 13, n)
    
    p  = 0.02
    p += 0.30 * (rainfall_mm / rainfall_mm.max())
    p += 0.25 * (river_level_m / 12)
    p += 0.15 * (1 - drainage_capacity)
    p += 0.12 * soil_saturation
    p += 0.10 * (upstream_flow / 5000)
    p += 0.08 * (1 - elevation / 500)
    p -= 0.05 * (distance_river_km / 20)
    p += np.where(np.isin(month, [6,7,8,9]), 0.08, 0)
    p += np.random.normal(0, 0.04, n)
    p  = p.clip(0, 1)
    flood = (p > np.percentile(p, 82)).astype(int)
    
    df = pd.DataFrame({
        'rainfall_mm':rainfall_mm.round(2),
        'river_level_m':river_level_m.round(2),
        'drainage_capacity':drainage_capacity.round(3),
        'soil_saturation':soil_saturation.round(3),
        'upstream_flow':upstream_flow.round(1),
        'elevation':elevation.round(1),
        'slope':slope.round(2),
        'distance_river_km':distance_river_km.round(3),
        'past_rain_3day':past_rain_3day.round(2),
        'past_rain_7day':past_rain_7day.round(2),
        'humidity':humidity.round(1),
        'month':month,
        'flood':flood
    })
    return df

df = generate_flood_data()
df.to_csv('flood_dataset.csv', index=False)
print(f"Dataset: {df.shape}")
print(f"Flood rate: {df['flood'].mean():.1%}")
df.head()

Dataset: (5000, 13)
Flood rate: 18.0%


,rainfall_mm,river_level_m,drainage_capacity,soil_saturation,upstream_flow,elevation,slope,distance_river_km,past_rain_3day,past_rain_7day,humidity,month,flood
0,57.38,2.43,0.381,0.410,3433.5,425.8,3.34,1.437,260.94,1479.55,62.4,1,0
1,35.16,2.68,0.260,0.415,121.4,118.5,7.06,0.947,70.74,391.24,99.8,2,0
2,62.67,5.80,0.873,0.589,279.7,300.2,13.84,0.606,278.51,742.00,58.8,3,0
3,93.31,3.79,0.463,0.445,2668.1,428.1,4.46,2.014,230.51,620.89,51.0,3,0
4,31.80,3.35,0.459,0.101,963.3,205.0,13.58,0.906,154.53,604.15,94.8,8,0


## Step 3 — Engineer + Train + Save

In [3]:
FEATURES = [
    'rainfall_mm','river_level_m','drainage_capacity','soil_saturation',
    'upstream_flow','elevation','slope','distance_river_km',
    'past_rain_3day','past_rain_7day','humidity','month',
    'rain_surge','flood_risk_index','drainage_stress'
]

def engineer(df):
    df = df.copy()
    df['rain_surge']       = df['rainfall_mm'] / (df['past_rain_3day'] + 1)
    df['flood_risk_index'] = df['river_level_m'] * df['soil_saturation']
    df['drainage_stress']  = (1 - df['drainage_capacity']) * df['rainfall_mm']
    return df

df_eng = engineer(df)
X = df_eng[FEATURES]; y = df_eng['flood']
X_tr,X_te,y_tr,y_te = train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)

model = RandomForestClassifier(n_estimators=200,max_depth=15,
    class_weight='balanced',random_state=42,n_jobs=-1)
model.fit(X_tr, y_tr)
pred  = model.predict(X_te)
proba = model.predict_proba(X_te)[:,1]

print("FLOOD MODEL RESULTS")
print(f"F1-Score : {f1_score(y_te, pred):.4f}")
print(f"ROC-AUC  : {roc_auc_score(y_te, proba):.4f}")
print()
print(classification_report(y_te, pred, target_names=['No Flood','Flood']))

FLOOD MODEL RESULTS
F1-Score : 0.7273
ROC-AUC  : 0.9606

              precision    recall  f1-score   support

    No Flood       0.92      0.98      0.95       820
       Flood       0.88      0.62      0.73       180

    accuracy                           0.92      1000
   macro avg       0.90      0.80      0.84      1000
weighted avg       0.91      0.92      0.91      1000



In [4]:
with open('flood_model.pkl','wb') as f:
    pickle.dump({'model':model,'features':FEATURES,'version':'1.0','disaster':'flood'},f)
print("flood_model.pkl saved! Copy to backend/ folder.")

flood_model.pkl saved! Copy to backend/ folder.
